# מעבדה 14 — דו-קיום פאזות

במעבדה הזו אתם פותחים מחדש את האיזותרמה המתפתלת של מודול 02 ומגלים במה הטבע מחליף את
הלולאה, ומדוע. אתם מאתרים את הקטע האסור, מעבירים את הקו השטוח דרך הלולאה בשתי דרכים שונות
ובודקים שהן מסכימות, בונים את עקומת הרתיחה נקודה אחר נקודה עד שהיא נגמרת, מוליכים נקודת מצב
לרוחב האזור הדו-פאזי, מחשבים את החום הכמוס פעמיים, מאמתים את קלאוזיוס–קלפרון נומרית, עוקפים
את הנקודה הקריטית, ומסיימים במדידת החום הכמוס של מים מעקומת לחץ האדים המפורסמת שלהם — עם
ביקורת על פס השגיאה.

המסלול: המודל והאובייקטים שלו (חלק 1); האנטומיה של איזותרמה אחת (חלק 2); בניית מקסוול, בשתי
דרכים (חלק 3); עקומת הדו-קיום מול פחמן דו-חמצני ממשי (חלק 4); כלל המנוף (חלק 5); חום כמוס
וקלאוזיוס–קלפרון (חלק 6); סביב הנקודה הקריטית (חלק 7); טבלת הקיטור (חלק 8); דיאגרמת הפאזות
הממשית וכלל הפאזות (חלק 9); הבדיקות האוטומטיות (חלק 10); וחקירה חופשית (חלק 11).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | זורם אחד של ואן דר ואלס — $N$ קבוע, קבועים $a$ ו-$b$ שהותאמו לנקודה הקריטית — המתואר על ידי $(P, v, T)$, שמתחת ל-$T_c$ מתפצל לנוזל ולגז החולקים $T$, $P$ ו-$\mu$ |
| **דינמיקה** | אין; כל מצב הוא מצב שיווי משקל והזזת $T$ פותרת מחדש את התנאים |
| **גבול** | סגור לחומר; $T$ (וכן $v$ הכולל או הלחץ) נקבעים מבחוץ, $P_{\mathrm{sat}}$ והנפחים המתקיימים יחד נפתרים |
| **צבר** | לא רלוונטי — תרמודינמיקה מאקרוסקופית על משוואת מצב של שדה ממוצע |
| **מוזנח** | משטחי מגע ומתח פנים, קינטיקה של גרעון, כל פאזה מוצקה, פלקטואציות ליד $T_c$ |
| **תקף כאשר** | צפיפויות מתונות, רכיב אחד, עבור *המבנה* של התשובה; ענפים מטא-יציבים הם טענות קיום בלבד |
| **אופני כישלון** | מעריכים של שדה ממוצע ליד $T_c$ (מודול 15), נוזלים עמוק מתחת לנקודה הקריטית, אין נקודה משולשת |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import gases, phases
from thermolab.constants import K_B, N_A, R_GAS
from thermolab.validation import convergence_study, relative_error

# Nothing in this laboratory is stochastic: every function in thermolab.phases is
# deterministic root-finding, so there is no generator to pass.

# Carbon dioxide's van der Waals constants from its measured critical point, exactly as in
# module 02's laboratory (NIST Chemistry WebBook: T_c = 304.13 K, P_c = 7.3773 MPa).
CO2_T_C, CO2_P_C = 304.13, 7.3773e6
A, B = gases.vdw_constants_from_critical(CO2_T_C, CO2_P_C)
V_C, T_C, P_C = gases.vdw_critical_point(A, B)
L_PER_MOL = N_A * 1e3  # m^3 per particle -> L/mol

T = 280.0  # K, the working temperature: 0.921 T_c
KT = K_B * T
print(f"a = {A:.4e} Pa m^6, b = {B:.4e} m^3 = {B * L_PER_MOL:.4f} L/mol")
print(f"v_c = {V_C * L_PER_MOL:.4f} L/mol, T_c = {T_C:.2f} K, P_c = {P_C / 1e6:.4f} MPa")
print(f"working temperature {T:.0f} K = {T / T_C:.4f} T_c;  k_B T = {KT:.4e} J")

## חלק 1 — האובייקטים

`thermolab.phases` פותר תנאי שיווי משקל; אין לו דינמיקה ואין לו מחולל אקראי.

| פונקציה | מה היא נותנת |
|---|---|
| `spinodal_volumes(T, a, b)` | שני הנפחים שבהם $(\partial P/\partial v)_T = 0$: המינימום והמקסימום של הלולאה |
| `maxwell_construction(T, a, b)` | $(P_{\mathrm{sat}}, v_\ell, v_g)$ בחצייה של שארית השטחים השווים |
| `coexistence_from_mu(T, a, b)` | אותם שלושה מספרים בחצייה של $\mu_\ell(P) - \mu_g(P)$ במקום זאת |
| `vdw_chemical_potential(v, T, a, b)` | $\mu(v, T)$ לאורך האיזותרמה, עד כדי פונקציה של $T$ |
| `coexistence_curve(T_grid, a, b)` | $P_{\mathrm{sat}}$, $v_\ell$, $v_g$ על רשת: הבינודל ועקומת הרתיחה |
| `lever_rule(v, v_l, v_g)` | $(x_\ell, x_g)$ למצב מעורב |
| `latent_heat`, `latent_heat_from_first_law` | $T(s_g - s_\ell)$ וכן $\Delta u + P_{\mathrm{sat}}\Delta v$ |
| `clausius_clapeyron_check(T, a, b)` | $\mathrm{d}P_{\mathrm{sat}}/\mathrm{d}T$ בהפרש מרכזי, וכן $L/(T\,\Delta v)$ |
| `equilibrium_volume(P, T, a, b)` | $v$ היציב ב-$(P, T)$: הפאזה בעלת $\mu$ הנמוך יותר |
| `clausius_clapeyron_fit(T, P)` | $\ln P = c - L/(\kB T)$ מותאם לנתונים, עם שגיאות ושאריות |
| `phase_rule(C, P)` | $F = C - P + 2$ |

הכול נפתר פנימית במשתנים מצומצמים $P/P_c$, $v/v_c$, $T/T_c$, ולכן התשובות לכל חומר של ואן דר
ואלס הן אוסף אחד של עקומות אוניברסליות; הקבועים רק מכיילים את הצירים.

### לנבא

התחייבו לתשובה לכל אחד מארבעת הניבויים של עמוד המודול לפני שתמשיכו:

1. היכן יושב הקו השטוח — בראש הלולאה, בתחתיתה, או במקום שכלל כלשהו כופה?
2. מים על הר בגובה 3000 מטר, ב-$0.7$ אטמוספרה: האם הם רותחים מעל, מתחת או ב-$100\,{}^\circ\mathrm{C}$?
3. לקומקום ברתיחה סוערת מכפילים את המבער. מה עושה המדחום?
4. מכל אטום מלא למחצה בנוזל ואדים מחומם אל מעבר ל-$T_c$. מה קורה?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — האנטומיה של איזותרמה אחת

פתחו מחדש את האיזותרמה של מודול 02 ב-$280\ \mathrm{K}$, הרבה מתחת ל-$T_c$. מדדו את השיפוע שלה
בהפרש מרכזי, הצללו את הקטע שבו השיפוע חיובי — המצבים שקריטריון היציבות של מודול 09 אוסר —
והשוו את שתי הנקודות שבהן השיפוע מחליף סימן לתנאי הספינודל בצורה סגורה
$\kB T = 2a(v - b)^2/v^3$.

### לנבא

האם המינימום של הלולאה יהיה בלחץ חיובי או שלילי בטמפרטורה הזו? (העמוד אומר שהמינימום חוצה
את האפס ב-$T = \tfrac{27}{32}\,T_c = 0.844\,T_c$.)

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
v_grid = np.geomspace(1.08 * B, 8.0 * V_C, 4000)
p_iso = gases.van_der_waals_pressure(v_grid, T, A, B)
h = 1e-4 * v_grid
slope = (gases.van_der_waals_pressure(v_grid + h, T, A, B)
         - gases.van_der_waals_pressure(v_grid - h, T, A, B)) / (2 * h)
forbidden = slope > 0

v_min, v_max = phases.spinodal_volumes(T, A, B)
p_min, p_max = gases.van_der_waals_pressure(np.array([v_min, v_max]), T, A, B)
# The closed-form condition, checked at both spinodal points.
condition = 2 * A * (np.array([v_min, v_max]) - B) ** 2 / np.array([v_min, v_max]) ** 3
spinodal_gap = float(np.max(np.abs(condition / KT - 1)))

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(v_grid * L_PER_MOL, p_iso / 1e6, color="k", lw=1.5, label="van der Waals, 280 K")
ax.fill_between(v_grid * L_PER_MOL, 0, p_iso / 1e6, where=forbidden, color="#fecaca",
                label="(dP/dv)_T > 0: forbidden")
ax.plot([v_min, v_max] * np.array(L_PER_MOL), [p_min / 1e6, p_max / 1e6], "o", color="#d97706",
        ms=8, label="spinodal points")
ax.set_xscale("log")
ax.set_xlim(0.04, 1.0)
ax.set_ylim(0, 8)
ax.set_xlabel("v (L/mol)")
ax.set_ylabel("P (MPa)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"slope changes sign on the grid at v = {v_grid[forbidden][0] * L_PER_MOL:.4f} and "
      f"{v_grid[forbidden][-1] * L_PER_MOL:.4f} L/mol")
print(f"closed-form spinodal volumes:      {v_min * L_PER_MOL:.4f} and "
      f"{v_max * L_PER_MOL:.4f} L/mol  ({v_min / V_C:.3f} and {v_max / V_C:.3f} v_c)")
print(f"k_B T = 2a(v-b)^2/v^3 there to {spinodal_gap:.1e}")
print(f"loop minimum {p_min / 1e6:.3f} MPa, maximum {p_max / 1e6:.3f} MPa")

שני לחצי הספינודל חיוביים ב-$280\ \mathrm{K}$: זה מעל $\tfrac{27}{32}\,T_c$.

## חלק 3 — בניית מקסוול, בשתי דרכים

שתי פאזות שמחליפות חלקיקים צריכות $\mu_\ell = \mu_g$. ב-$T$ קבוע, גיבס–דוהם נותן
$\mathrm{d}\mu = v\,\mathrm{d}P$, ובאינטגרציה לאורך איזותרמת המודל מ-$v_\ell$ עד $v_g$,

$$
\mu_g - \mu_\ell = \int_\ell^g v\,\mathrm{d}P = P_{\mathrm{sat}}\,(v_g - v_\ell) - \int_{v_\ell}^{v_g} P\,\mathrm{d}v .
$$

השוואה לאפס היא תנאי השטחים השווים. `maxwell_construction` חוצה את הפרש השטחים;
`coexistence_from_mu` חוצה את $\mu_\ell(P) - \mu_g(P)$ מתוך $\mu$ בצורה סגורה ולעולם אינה מחשבת
שטח. אם המשפט הוא משפט, הן מסכימות.

### לנבא

היכן יושב הקו השטוח ביחס למינימום של הלולאה ($4.12$ MPa) ולמקסימום שלה
($5.69$ MPa)? כתבו מספר.

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
p_sat, v_l, v_g = phases.maxwell_construction(T, A, B)
p_mu, v_l_mu, v_g_mu = phases.coexistence_from_mu(T, A, B)

# The two lobes, by a fine trapezoid rule that knows nothing about the solver.
v_lobes = np.linspace(v_l, v_g, 200_001)
excess = gases.van_der_waals_pressure(v_lobes, T, A, B) - p_sat
dv = np.diff(v_lobes)
lobe_above = float(np.sum(np.where(excess > 0, excess, 0.0)[:-1] * dv))
lobe_below = float(-np.sum(np.where(excess < 0, excess, 0.0)[:-1] * dv))

# mu along the isotherm, shifted so the coexistence value is zero.
mu_iso = phases.vdw_chemical_potential(v_grid, T, A, B)
mu_sat = phases.vdw_chemical_potential(v_l, T, A, B)
mu_gap = (mu_sat - phases.vdw_chemical_potential(v_g, T, A, B)) / KT

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))
left.plot(v_grid * L_PER_MOL, p_iso / 1e6, color="k", lw=1.5)
inside = (v_grid >= v_l) & (v_grid <= v_g)
left.fill_between(v_grid[inside] * L_PER_MOL, p_iso[inside] / 1e6, p_sat / 1e6,
                  where=p_iso[inside] <= p_sat, color="#bfdbfe")
left.fill_between(v_grid[inside] * L_PER_MOL, p_iso[inside] / 1e6, p_sat / 1e6,
                  where=p_iso[inside] >= p_sat, color="#fecaca")
left.plot([v_l * L_PER_MOL, v_g * L_PER_MOL], [p_sat / 1e6] * 2, color="#dc2626", lw=2.5,
          label=f"P_sat = {p_sat / 1e6:.3f} MPa")
left.set_xscale("log")
left.set_xlim(0.04, 1.0)
left.set_ylim(3, 7)
left.set_xlabel("v (L/mol)")
left.set_ylabel("P (MPa)")
left.legend()
keep = (p_iso > 2e6) & (p_iso < 8e6)
right.plot(p_iso[keep] / 1e6, (mu_iso[keep] - mu_sat) / KT, color="k", lw=1.5)
right.axhline(0, color="grey", lw=0.8)
right.plot([p_sat / 1e6], [0], "o", color="#dc2626", ms=8, label="crossing = P_sat")
right.set_xlabel("P (MPa)")
right.set_ylabel("(mu - mu_sat) / k_B T")
right.legend()
plt.tight_layout()
plt.show()

print(f"equal-area:  P_sat = {p_sat / 1e6:.6f} MPa, v_l = {v_l * L_PER_MOL:.5f}, "
      f"v_g = {v_g * L_PER_MOL:.5f} L/mol")
print(f"equal-mu:    P_sat = {p_mu / 1e6:.6f} MPa, v_l = {v_l_mu * L_PER_MOL:.5f}, "
      f"v_g = {v_g_mu * L_PER_MOL:.5f} L/mol")
print(f"relative differences: {relative_error(p_sat, p_mu):.1e}, "
      f"{relative_error(v_l, v_l_mu):.1e}, {relative_error(v_g, v_g_mu):.1e}")
print(f"lobe above the line: {lobe_above:.6e} J/particle = {lobe_above / KT:.4f} k_B T")
print(f"lobe below the line: {lobe_below:.6e} J/particle;  relative difference "
      f"{abs(lobe_above - lobe_below) / lobe_above:.1e}")
print(f"mu_l - mu_g at P_sat: {mu_gap:.1e} k_B T")
print(f"the line sits {(p_sat - p_min) / (p_max - p_min):.0%} of the way from the loop's "
      "minimum to its maximum")

שטחים שווים ו-$\mu$ שווה נותנים את אותו לחץ עד כדי עיגול, ובלחץ הזה שני הפוטנציאלים הכימיים
מסכימים עד כדי $10^{-15}\,\kB T$. הבנייה *היא* המשפט, מחושב.

## חלק 4 — עקומת הדו-קיום, ופחמן דו-חמצני ממשי

פתרו את הבנייה על רשת של טמפרטורות עד $T_c$. שני קצות הקו השטוח משרטטים את כיפת הבינודל
במישור $P$–$v$; הגובה שלו משרטט את עקומת הרתיחה במישור $P$–$T$. אחר כך הניחו מעל זה את עקומת
הרוויה *המדודה* של פחמן דו-חמצני, המצורפת כ-`data/14-co2-saturation.csv`.

### לנבא

האם עקומת הרתיחה של המודל תשכב מעל המדודה, מתחתיה, או עליה? האם הפער יגדל או יקטן לעבר
$T_c$?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    DATA = Path("..", "..", "..", "data")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSVs co-located here.
    DATA = Path("data")

co2_t, co2_p = np.genfromtxt(DATA / "14-co2-saturation.csv", delimiter=",", comments="#").T

t_grid = np.linspace(0.5 * T_C, T_C, 200)
p_curve, vl_curve, vg_curve = phases.coexistence_curve(t_grid, A, B)
sp_t, sp_v, sp_p = phases.spinodal_curve(A, B, n_points=200, t_min=0.5)

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))
left.fill_betweenx(p_curve / 1e6, vl_curve * L_PER_MOL, vg_curve * L_PER_MOL, color="#f1f5f9")
left.plot(vl_curve * L_PER_MOL, p_curve / 1e6, color="#64748b", lw=1.5, label="binodal")
left.plot(vg_curve * L_PER_MOL, p_curve / 1e6, color="#64748b", lw=1.5)
left.plot(sp_v * L_PER_MOL, sp_p / 1e6, ":", color="#d97706", lw=1.5, label="spinodal")
left.plot(v_grid * L_PER_MOL, p_iso / 1e6, color="k", lw=1, label="280 K isotherm")
left.plot([v_l * L_PER_MOL, v_g * L_PER_MOL], [p_sat / 1e6] * 2, color="#dc2626", lw=2)
left.set_xscale("log")
left.set_xlim(0.04, 3.0)
left.set_ylim(0, 8)
left.set_xlabel("v (L/mol)")
left.set_ylabel("P (MPa)")
left.legend()
right.plot(t_grid, p_curve / 1e6, color="#dc2626", lw=2, label="van der Waals")
right.plot(co2_t, co2_p / 1e6, "o", color="#2563eb", ms=4, label="CO2, measured curve")
right.plot([T_C], [P_C / 1e6], "o", mfc="white", mec="k", ms=9, label="critical point")
right.set_xlim(210, 320)
right.set_xlabel("T (K)")
right.set_ylabel("P_sat (MPa)")
right.legend()
plt.tight_layout()
plt.show()

ratios = {}
for t_check in (216.592, 250.0, 280.0, 300.0):
    model = phases.maxwell_construction(t_check, A, B)[0]
    measured = np.interp(t_check, co2_t, co2_p)
    ratios[t_check] = model / measured
    print(f"T = {t_check:7.3f} K ({t_check / T_C:.3f} T_c): model {model / 1e6:.4f} MPa, "
          f"measured {measured / 1e6:.4f} MPa, ratio {model / measured:.3f}")
print(f"the model curve ends at ({t_grid[-1]:.2f} K, {p_curve[-1] / 1e6:.4f} MPa) = (T_c, P_c)")

עקומת המודל גבוהה מדי בכל מקום והשגיאה גדלה ככל ש-$T$ יורד — פי $3$ בנקודה המשולשת.
*הצורה* נכונה: עקומה אחת, עולה, שנגמרת בנקודה הקריטית.

## חלק 5 — כלל המנוף

הוליכו את הנפח הכולל לחלקיק לרוחב הקטע השטוח ב-$280\ \mathrm{K}$. בכל $v$ הזורם הוא שבר
$x_\ell$ נוזל ב-$v_\ell$ ושבר $x_g$ גז ב-$v_g$, עם
$x_g = (v - v_\ell)/(v_g - v_\ell)$.

### לנבא

באמצע הדרך לרוחב הקטע, חצי מהמולקולות הן אדים. איזה שבר מ*הנפח* הוא נוזל?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
v_walk = np.linspace(v_l, v_g, 201)
x_liq, x_gas = phases.lever_rule(v_walk, v_l, v_g)
liquid_volume_fraction = x_liq * v_l / v_walk

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(v_walk * L_PER_MOL, x_liq, color="#2563eb", lw=2, label="x_liq (particles)")
ax.plot(v_walk * L_PER_MOL, x_gas, color="#d97706", lw=2, label="x_gas (particles)")
ax.plot(v_walk * L_PER_MOL, liquid_volume_fraction, "--", color="#2563eb", lw=1.5,
        label="liquid share of the volume")
ax.set_xlabel("v (L/mol)")
ax.set_ylabel("fraction")
ax.legend()
plt.tight_layout()
plt.show()

v_half = 0.5 * (v_l + v_g)
x_l_half, x_g_half = phases.lever_rule(v_half, v_l, v_g)
print(f"halfway, v = {v_half * L_PER_MOL:.4f} L/mol: x_liq = {x_l_half:.3f}, "
      f"x_gas = {x_g_half:.3f}, liquid occupies {x_l_half * v_l / v_half:.1%} of the volume")
# A rigid 1 L cylinder holding 5 mol: v = 0.2 L/mol.
v_cyl = 0.2e-3 / N_A
x_l_cyl, x_g_cyl = phases.lever_rule(v_cyl, v_l, v_g)
closure = float(np.max(np.abs(x_liq + x_gas - 1)))
roundtrip = float(np.max(np.abs((x_liq * v_l + x_gas * v_g) / v_walk - 1)))
print(f"1 L cylinder with 5 mol at 280 K: x_gas = {x_g_cyl:.3f} (model volumes)")
print(f"x_liq + x_gas - 1: {closure:.1e};  x_liq v_l + x_gas v_g against v: {roundtrip:.1e}")

## חלק 6 — חום כמוס וקלאוזיוס–קלפרון

שתי דרכים לחום הכמוס: $L = T(s_g - s_\ell) = \kB T\ln[(v_g - b)/(v_\ell - b)]$ מהאנטרופיות,
ו-$L = \Delta u + P_{\mathrm{sat}}\,\Delta v$ מהחוק הראשון לאורך הקו השטוח. הן שוות בדיוק
ב-$P_{\mathrm{sat}}$ (בעיה 2 באוסף הבעיות מראה שההפרש ביניהן הוא $\mu_g - \mu_\ell$). ואז
קלאוזיוס–קלפרון: השיפוע של עקומת הרתיחה בהפרש מרכזי כנגד $L/(T\,\Delta v)$.

### לנבא

כאשר $T \to T_c$, האם $L$ עולה, יורד, או נשאר במקומו? מה קורה לשיפוע
$\mathrm{d}P_{\mathrm{sat}}/\mathrm{d}T$ שם?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
t_latent = np.linspace(0.5 * T_C, phases.T_RATIO_MAX * T_C, 120)
L_entropy = phases.latent_heat(t_latent, A, B)
L_first_law = phases.latent_heat_from_first_law(t_latent, A, B)
latent_gap = float(np.max(np.abs(L_entropy / L_first_law - 1)))

numeric, predicted = phases.clausius_clapeyron_check(T, A, B)
order = convergence_study(lambda k: phases.clausius_clapeyron_check(T, A, B, dt=2.0 / k)[0],
                          [1, 2, 4, 8, 16], predicted)

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(t_latent, L_entropy * N_A / 1e3, color="#dc2626", lw=2, label="T (s_g - s_l)")
left.plot(t_latent, L_first_law * N_A / 1e3, "--", color="k", lw=1.2,
          label="Delta u + P_sat Delta v")
left.axvline(T_C, color="grey", ls=":")
left.set_xlabel("T (K)")
left.set_ylabel("L (kJ/mol)")
left.legend()
dp_dt = np.gradient(p_curve, t_grid)
_, vl_latent, vg_latent = phases.maxwell_construction(t_latent, A, B)
right.plot(t_grid, dp_dt / 1e3, color="k", lw=1.5, label="dP_sat/dT, numerical")
right.plot(t_latent, L_entropy / (t_latent * (vg_latent - vl_latent)) / 1e3,
           "--", color="#dc2626", lw=1.5, label="L / (T Delta v)")
right.set_xlabel("T (K)")
right.set_ylabel("dP_sat/dT (kPa/K)")
right.legend()
plt.tight_layout()
plt.show()

print(f"L at 280 K: {phases.latent_heat(T, A, B) * N_A / 1e3:.4f} kJ/mol = "
      f"{phases.latent_heat(T, A, B) / KT:.4f} k_B T per molecule")
print(f"largest relative gap between the two routes over the grid: {latent_gap:.1e}")
print(f"L at {t_latent[-1] / T_C:.5f} T_c: {L_entropy[-1] * N_A:.3f} J/mol  (it dies at T_c)")
print(f"Clausius-Clapeyron at 280 K: numerical {numeric / 1e3:.6f} kPa/K, "
      f"L/(T Delta v) {predicted / 1e3:.6f} kPa/K, "
      f"relative gap {relative_error(numeric, predicted):.1e}")
print(f"observed order of the central difference: {order.observed_order:.3f}")
# The real latent heat of CO2 at 280 K, from the measured curve and module 02's plateau volumes:
dp_real = (np.interp(280.5, co2_t, co2_p) - np.interp(279.5, co2_t, co2_p)) / 1.0
dv_real = 3.834e-4 - 5.32e-5  # m^3/mol, from co2-isotherm-280k.csv
print(f"real CO2: dP_sat/dT = {dp_real / 1e3:.1f} kPa/K, Delta v = {dv_real:.3e} m^3/mol -> "
      f"L = {280.0 * dv_real * dp_real / 1e3:.2f} kJ/mol (model: "
      f"{phases.latent_heat(T, A, B) * N_A / 1e3:.2f})")

## חלק 7 — סביב הנקודה הקריטית

שני מסלולים במישור $P$–$T$ מאותו מצב גז לאותו מצב נוזל: אחד ישר לרוחב עקומת הדו-קיום
ב-$0.85\,T_c$, אחד סביב קצה שלה — החוצה אל מעבר ל-$T_c$, למעלה אל מעבר ל-$P_c$, וחזרה.
`equilibrium_volume` מחזירה את הפאזה היציבה בכל $(P, T)$.

In [ ]:
t_a, p_lo, p_hi, t_far = 0.85, 0.30, 1.15, 1.20
s = np.linspace(0.0, 1.0, 600)
path_across = np.column_stack([np.full_like(s, t_a), p_lo + (p_hi - p_lo) * s])
leg = np.clip(3.0 * s[:, None] - np.arange(3)[None, :], 0.0, 1.0)
path_around = np.column_stack([t_a + (t_far - t_a) * (leg[:, 0] - leg[:, 2]),
                               p_lo + (p_hi - p_lo) * leg[:, 1]])


def density_along(path):
    return V_C / phases.equilibrium_volume(path[:, 1] * P_C, path[:, 0] * T_C, A, B)


rho_across, rho_around = density_along(path_across), density_along(path_around)
jump_across = float(np.max(np.abs(np.diff(rho_across))))
jump_around = float(np.max(np.abs(np.diff(rho_around))))

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(t_grid / T_C, p_curve / P_C, color="k", lw=2, label="coexistence curve")
left.plot(path_across[:, 0], path_across[:, 1], ":", color="#dc2626", lw=2, label="across")
left.plot(path_around[:, 0], path_around[:, 1], ":", color="#2563eb", lw=2, label="around")
left.plot([1], [1], "o", mfc="white", mec="k", ms=9)
left.set_xlabel("T / T_c")
left.set_ylabel("P / P_c")
left.legend()
right.plot(s, rho_across, color="#dc2626", lw=2, label="across")
right.plot(s, rho_around, color="#2563eb", lw=2, label="around")
right.set_xlabel("path parameter s")
right.set_ylabel("rho / rho_c")
right.legend()
plt.tight_layout()
plt.show()

print(f"both paths start at rho/rho_c = {rho_across[0]:.3f} and end at {rho_across[-1]:.3f}")
print(f"largest step in density between neighbouring points: across {jump_across:.3f}, "
      f"around {jump_around:.4f}")

המסלול שחוצה את העקומה קופץ פי שישה בצעד אחד — עיבוי. המסלול סביב הנקודה הקריטית משנה את
הצפיפות באותה מידה, בלי קפיצה בשום מקום: שום דבר אינו אוזל ב-$T_c$, ושתי הפאזות פשוט מפסיקות
להיות שונות.

## חלק 8 — טבלת הקיטור: מדידה של החום הכמוס של מים

`data/14-water-saturation.csv` מכיל את לחץ הרוויה המפורסם של מים מ-$275$ עד
$425\ \mathrm{K}$ — המשוואה שמאחורי כל טבלת קיטור. אם האדים אידיאליים, $v_\ell \ll v_g$
ו-$L$ קבוע, אז $\ln P_{\mathrm{sat}} = c - L/(\kB T)$, והתאמת קו ישר של
$\ln P$ כנגד $1/T$ מודדת את $L$.

> **על הנתונים.** השורות הן משוואת לחץ הרוויה של IAPWS המחושבת כל 5 K ומעוגלת לשש ספרות —
> ערכי ייחוס שמשחזרים את טבלאות הקיטור המדודות עד כדי $0.01\%$, לא מדידה שנעשתה לקורס הזה.
> הכותרת של הקובץ אומרת בדיוק כיצד הוא נוצר.

### לנבא

האם התאמה על כל התחום, או על חלון צר סביב $373\ \mathrm{K}$, תיתן חום כמוס קרוב יותר
ל-$40.66\ \mathrm{kJ/mol}$ שבטבלאות ב-$100\,{}^\circ\mathrm{C}$? מדוע?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
water_t, water_p = np.genfromtxt(DATA / "14-water-saturation.csv", delimiter=",", comments="#").T
KJ_PER_MOL = N_A / 1e3
MJ_PER_KG = N_A / 18.015e-3 / 1e6

fit_all = phases.clausius_clapeyron_fit(water_t, water_p)
narrow = (water_t >= 345) & (water_t <= 405)
fit_narrow = phases.clausius_clapeyron_fit(water_t[narrow], water_p[narrow])
cold = water_t <= 325
fit_cold = phases.clausius_clapeyron_fit(water_t[cold], water_p[cold])

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(1e3 / water_t, np.log(water_p), "o", color="#2563eb", ms=4, label="steam table")
left.plot(1e3 / water_t, np.log(fit_all.pressure(water_t)), color="k", lw=1,
          label=f"fit, all: L = {fit_all.latent_heat * KJ_PER_MOL:.2f} kJ/mol")
left.set_xlabel("1000 / T  (1/K)")
left.set_ylabel("ln(P_sat / Pa)")
left.legend()
right.plot(water_t, fit_all.residuals, "o-", color="k", ms=4, label="all 31 points")
right.plot(water_t[narrow], fit_narrow.residuals, "s-", color="#dc2626", ms=4, label="345-405 K")
right.axhline(0, color="grey", lw=0.8)
right.set_xlabel("T (K)")
right.set_ylabel("ln P_data - ln P_fit")
right.legend()
plt.tight_layout()
plt.show()

fits = (("all 275-425 K", fit_all), ("345-405 K", fit_narrow), ("275-325 K", fit_cold))
for name, fit in fits:
    print(f"{name:>14}: L = {fit.latent_heat * KJ_PER_MOL:.2f} +/- "
          f"{fit.latent_heat_error * KJ_PER_MOL:.2f} kJ/mol = {fit.latent_heat * MJ_PER_KG:.3f} "
          f"MJ/kg;  boils at 1 atm at {fit.boiling_temperature(101325.0):.2f} K;  "
          f"max |residual| {np.abs(fit.residuals).max():.4f}")
print("tabulated: L = 40.66 kJ/mol = 2.257 MJ/kg at 100 C; boiling point 373.12 K")
for atm in (0.33, 0.7, 0.83, 2.0):
    t_boil = fit_narrow.boiling_temperature(atm * 101325.0)
    print(f"  at {atm:.2f} atm water boils at {t_boil:.1f} K = {t_boil - 273.15:.0f} C")
# Two neighbouring rows alone: the local slope at 372.5 K.
i = int(np.searchsorted(water_t, 370.0))
L_two_point = R_GAS * np.log(water_p[i + 1] / water_p[i]) / (1 / water_t[i] - 1 / water_t[i + 1])
print(f"two-point estimate from {water_t[i]:.0f} and {water_t[i + 1]:.0f} K: "
      f"L = {L_two_point / 1e3:.2f} kJ/mol")

ההתאמה הרחבה מוטה כלפי מעלה מפני ש-$L$ יורד עם הטמפרטורה — השאריות מתעקלות במקום להתפזר,
וזהו הסימן של מודל שגוי באופן שיטתי, לא רועש. השגיאות המצוטטות הן הסטטיסטיות ואינן יודעות
דבר על כך.

## חלק 9 — דיאגרמת הפאזות הממשית, וכלל הפאזות

למודל ואן דר ואלס אין מוצק, ולכן כלל הפאזות מודגם על הגבולות הממשיים של מים: המראה, התכה
ואידוי מתוך `data/14-water-phase-boundaries.csv`, הנפגשים בנקודה המשולשת.

In [ ]:
rows = np.genfromtxt(DATA / "14-water-phase-boundaries.csv", delimiter=",", comments="#")
boundary, bt, bp = rows[:, 0].astype(int), rows[:, 1], rows[:, 2]
A_W, B_W = gases.vdw_constants_from_critical(647.096, 22.064e6)
t_model_w = np.linspace(0.45 * 647.096, phases.T_RATIO_MAX * 647.096, 200)
p_model_w, _, _ = phases.coexistence_curve(t_model_w, A_W, B_W)

fig, ax = plt.subplots(figsize=(7, 4.5))
boundaries = ((0, "#94a3b8", "sublimation"), (1, "#2563eb", "melting"),
              (2, "#dc2626", "vaporization"))
for which, color, name in boundaries:
    sel = boundary == which
    ax.plot(bt[sel], bp[sel], color=color, lw=2, label=name)
ax.plot(t_model_w, p_model_w, "--", color="#dc2626", lw=1,
        label="van der Waals, from water's T_c, P_c")
ax.plot([273.16], [611.657], "o", color="k", label="triple point")
ax.plot([647.096], [22.064e6], "o", mfc="white", mec="k", ms=8, label="critical point")
ax.set_yscale("log")
ax.set_xlabel("T (K)")
ax.set_ylabel("P (Pa)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

triple_ok = all(abs(np.interp(273.16, bt[boundary == k], bp[boundary == k]) / 611.657 - 1) < 1e-3
                for k in (0, 1, 2))
print("the three boundaries meet at the triple point:", triple_ok)
features = ((1, 1, "an area"), (1, 2, "a curve"), (1, 3, "the triple point"),
            (2, 2, "salt water and its vapour"))
for components, n_phases, feature in features:
    print(f"C = {components}, P = {n_phases}: F = {phases.phase_rule(components, n_phases)}"
          f"  ({feature})")
p_boil_model, _, _ = phases.maxwell_construction(373.124, A_W, B_W)
print(f"van der Waals water at 373.1 K: P_sat = {p_boil_model / 1e3:.0f} kPa against 101 kPa;"
      f" L = {phases.latent_heat(373.124, A_W, B_W) * KJ_PER_MOL:.1f} kJ/mol against 40.7")

## חלק 10 — בדיקות אוטומטיות

חישוב שלא בדקתם הוא תמונה, לא ראיה. אלה אותן טענות שרצות בחבילת הבדיקות של הפרויקט.

In [ ]:
# 1. The spinodal points satisfy the closed-form condition (Part 2).
assert spinodal_gap < 1e-9
assert v_l < v_min < v_max < v_g

# 2. Equal areas is equal mu: the two solvers agree, the lobes match, mu_l = mu_g (Part 3).
assert relative_error(p_sat, p_mu) < 1e-12
assert relative_error(v_l, v_l_mu) < 1e-8 and relative_error(v_g, v_g_mu) < 1e-8
assert abs(lobe_above - lobe_below) / lobe_above < 1e-8
assert abs(mu_gap) < 1e-12

# 3. The coexistence curve ends at the critical point, and the model is high everywhere (Part 4).
assert relative_error(p_curve[-1], P_C) < 1e-12 and relative_error(vl_curve[-1], V_C) < 1e-12
assert all(r > 1.0 for r in ratios.values())

# 4. The lever rule closes exactly (Part 5).
assert closure < 1e-15 and roundtrip < 1e-14

# 5. Two latent heats agree, L dies at T_c, Clausius-Clapeyron holds to second order (Part 6).
assert latent_gap < 1e-10
assert L_entropy[-1] < 1e-2 * L_entropy[0]
assert relative_error(numeric, predicted) < 1e-8
assert abs(order.observed_order - 2.0) < 0.05

# 6. Across the curve the density jumps; around the critical point it never does (Part 7).
assert jump_across > 1.0 and jump_around < 0.05
assert relative_error(rho_across[-1], rho_around[-1]) < 1e-10

# 7. The steam-table fit lands within 5% of the tabulated latent heat, the narrow one within
#    2% (Part 8).
assert abs(fit_all.latent_heat * KJ_PER_MOL / 40.66 - 1) < 0.05
assert abs(fit_narrow.latent_heat * KJ_PER_MOL / 40.66 - 1) < 0.02
assert abs(fit_narrow.boiling_temperature(101325.0) - 373.12) < 0.5

# 8. The real boundaries meet at the triple point, and the phase rule counts (Part 9).
assert triple_ok
assert phases.phase_rule(1, 3) == 0

print("all checks passed")

## חלק 11 — חקרו בעצמכם

קבעו את הטמפרטורה המצומצמת ונפח כולל, בחרו מה להניח מעל, ואז לחצו על **Run Interact**. הלוח
השמאלי מראה את האיזותרמה עם הקו השטוח והאונות המוצללות שלה, את נקודת המצב בנפח שבחרתם,
ולפי בחירה את כיפות הבינודל והספינודל; הלוח הימני מראה את לולאת $\mu$–$P$. שלושה דברים ששווה
לנסות:

1. דחפו את $T/T_c$ לעבר $0.9999$ וראו את האונות, את הקו ואת לולאת $\mu$ קורסים.
2. קבעו $T/T_c$ מתחת ל-$0.844$ ומצאו את המינימום של הלולאה מתחת ללחץ אפס. היכן נמצא עכשיו
   הקו השטוח ביחס ללולאה?
3. הציבו את נקודת המצב ממש מחוץ לקו השטוח בצד הנוזל, ואז ממש בתוכו. מה משתנה בפלט, ומה
   לא?

In [ ]:
import ipywidgets as widgets


def explore(t_ratio=0.9, v_ratio=1.5, show_domes=True, show_mu_loop=True):
    t_here = t_ratio * T_C
    v_here = v_ratio * V_C
    p_here, vl_here, vg_here = phases.maxwell_construction(t_here, A, B)
    v_plot = np.geomspace(1.05 * B, 60 * V_C, 3000)
    p_plot = gases.van_der_waals_pressure(v_plot, t_here, A, B)
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
    if show_domes:
        left.fill_betweenx(p_curve / P_C, vl_curve / V_C, vg_curve / V_C, color="#f1f5f9")
        left.plot(vl_curve / V_C, p_curve / P_C, color="#64748b", lw=1)
        left.plot(vg_curve / V_C, p_curve / P_C, color="#64748b", lw=1)
        left.plot(sp_v / V_C, sp_p / P_C, ":", color="#d97706", lw=1.2)
    left.plot(v_plot / V_C, p_plot / P_C, color="k", lw=1.5)
    inside = (v_plot >= vl_here) & (v_plot <= vg_here)
    left.fill_between(v_plot[inside] / V_C, p_plot[inside] / P_C, p_here / P_C,
                      where=p_plot[inside] <= p_here, color="#bfdbfe")
    left.fill_between(v_plot[inside] / V_C, p_plot[inside] / P_C, p_here / P_C,
                      where=p_plot[inside] >= p_here, color="#fecaca")
    left.plot([vl_here / V_C, vg_here / V_C], [p_here / P_C] * 2, color="#dc2626", lw=2.5)
    if vl_here <= v_here <= vg_here:
        x_l_here, x_g_here = phases.lever_rule(v_here, vl_here, vg_here)
        p_state = p_here
        state = f"two phases: x_liq = {x_l_here:.3f}, x_gas = {x_g_here:.3f}"
    else:
        p_state = gases.van_der_waals_pressure(v_here, t_here, A, B)
        state = "one phase: " + ("liquid" if v_here < vl_here else "gas")
    left.plot([v_ratio], [p_state / P_C], "o", color="#2563eb", ms=10)
    left.set_xscale("log")
    left.set_xlim(0.35, 60)
    left.set_ylim(min(-0.5, 1.1 * float(p_plot.min()) / P_C), 1.2)
    left.set_xlabel("v / v_c")
    left.set_ylabel("P / P_c")
    if show_mu_loop:
        mu_plot = phases.vdw_chemical_potential(v_plot, t_here, A, B)
        mu_ref = phases.vdw_chemical_potential(vl_here, t_here, A, B)
        keep_plot = (p_plot > 0) & (p_plot < 1.3 * P_C)
        right.plot(p_plot[keep_plot] / P_C, (mu_plot[keep_plot] - mu_ref) / (P_C * V_C),
                   color="k")
        right.axhline(0, color="grey", lw=0.8)
        right.plot([p_here / P_C], [0], "o", color="#dc2626", ms=8)
        right.set_xlabel("P / P_c")
        right.set_ylabel("(mu - mu_sat) / (P_c v_c)")
    else:
        right.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"T = {t_here:.2f} K: P_sat = {p_here / 1e6:.4f} MPa = {p_here / P_C:.4f} P_c;  "
          f"v_l = {vl_here / V_C:.4f} v_c, v_g = {vg_here / V_C:.4f} v_c;  "
          f"L = {phases.latent_heat(t_here, A, B) * N_A / 1e3:.3f} kJ/mol")
    print(f"state at v = {v_ratio} v_c: P = {p_state / 1e6:.4f} MPa;  {state}")


widgets.interact_manual(
    explore,
    t_ratio=widgets.FloatSlider(value=0.9, min=0.5, max=0.9999, step=0.0001,
                                readout_format=".4f", description="T / T_c"),
    v_ratio=widgets.FloatLogSlider(value=1.5, base=10, min=-0.4, max=1.5, step=0.01,
                                   description="v / v_c"),
    show_domes=widgets.Checkbox(value=True, description="binodal + spinodal"),
    show_mu_loop=widgets.Checkbox(value=True, description="mu-P loop"),
);

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון בבדיקה אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בעמוד המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "14-coexistence.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שתלכו

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהנמקה שלכם?
2. בחלק 3 שני פותרים שאינם חולקים שום נוסחה נתנו את אותו לחץ. איזה משפט ההסכמה הזו מחשבת,
   והיכן נכנס בו הפוטנציאל הכימי?
3. ההתאמה הרחבה וההתאמה הצרה של חלק 8 נבדלו בפי שלושים מהשגיאות המצוטטות שלהן. מה שגיאת
   התקן של התאמה מודדת, ומה היא אינה מודדת?
4. המסלול הכחול של חלק 7 הפך גז לנוזל בלי קפיצה. מה, אם כן, ההבדל בין נוזל לגז?

**התשובות שלכם:**

1.
2.
3.
4.